# Glue ETL: Silver → Gold Feature Aggregation
This notebook implements the Glue ETL process that aggregates Silver event data into Gold user-level features.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta


In [ ]:
# Glue Job execution time 기준 전일 데이터 처리
TARGET_DATE = (datetime.utcnow() - timedelta(days=1)).strftime('%Y-%m-%d')

print(f"[INFO] TARGET_DATE = {TARGET_DATE}")

SILVER_PATH = f"s3://ameowzon-silver/events/event_date={TARGET_DATE}/"
GOLD_PATH   = f"s3://ameowzon-gold/events/event_date={TARGET_DATE}/"

print("[INFO] SILVER_PATH:", SILVER_PATH)
print("[INFO] GOLD_PATH  :", GOLD_PATH)


In [ ]:
df = spark.read.parquet(SILVER_PATH)


## Basic column normalization


In [ ]:
df = (
    df
    .withColumn("event_ts", F.col("event_ts").cast("long"))
    .withColumn(
        "event_time",
        F.to_timestamp(F.col("event_ts") / 1000)
    )
)


## User behavior aggregation features


In [ ]:
basic_features = (
    df.groupBy("user_id")
    .agg(
        F.count(F.when(
            F.col("event_name").isin("SCREEN_INTERACTIVE","SCREEN_NON_INTERACTIVE"),True)).alias("Screen"),

        F.count(F.when(
            F.col("event_name") == "USER_INTERACTION",True)).alias("UserAct"),

        F.count(F.when(
            F.col("event_name") == "NOTIFICATION_INTERRUPTION",True)).alias("Notif"),

        F.count(F.when(
            F.col("event_name") == "KEYGUARD_HIDDEN",True)).alias("unlock_cnt"),

        F.count(F.when(
            F.col("event_type") == "heartbeat",True)).alias("heartbeat_cnt"),

        F.max("retry_count").alias("retry_max"),
        F.max("queue_size").alias("queue_max"),

        F.min("tz_offset_minutes").alias("tz_offset_min"),
        F.max("tz_offset_minutes").alias("tz_offset_max"),

        F.max("client_last_event_ts").alias("qc_last_ts_max"),

        F.sum("step_count").alias("step_sum")
    )
)


## Mobility features: Cell tower changes


In [ ]:
cell_w = Window.partitionBy("user_id").orderBy("event_ts")

df_cell = (
    df
    .withColumn("prev_cell", F.lag("cell_lac").over(cell_w))
    .withColumn(
        "cell_changed",
        F.when(
            (F.col("cell_lac").isNotNull()) &
            (F.col("prev_cell").isNotNull()) &
            (F.col("cell_lac") != F.col("prev_cell")),
            1
        ).otherwise(0)
    )
)


In [ ]:
cell_features = (
    df_cell.groupBy("user_id")
    .agg(
        F.sum("cell_changed").alias("cell_change_cnt"),
        F.countDistinct("cell_lac").alias("unique_cell_cnt")
    )
)


## WiFi environment change features


In [ ]:
wifi_features = (
    df
    .filter(F.col("_activity") == "WIFI_CHANGE")
    .groupBy("user_id")
    .agg(
        F.count("*").alias("wifi_change_cnt_est"),

        F.countDistinct(
            F.when(
                (F.col("curr_bssid").isNotNull()) &
                (F.col("curr_bssid") != "none"),
                F.col("curr_bssid")
            )
        ).alias("unique_wifi_cnt")
    )
)


## Feature merge


In [ ]:
features = (
    basic_features
    .join(cell_features, "user_id", "left")
    .join(wifi_features, "user_id", "left")
)


## Null handling


In [ ]:
features = features.fillna({
    "Screen": 0,
    "UserAct": 0,
    "Notif": 0,
    "unlock_cnt": 0,
    "heartbeat_cnt": 0,
    "retry_max": 0,
    "queue_max": 0,
    "cell_change_cnt": 0,
    "unique_cell_cnt": 0,
    "wifi_change_cnt_est": 0,
    "unique_wifi_cnt": 0,
    "step_sum": 0
})


## Save Gold dataset


In [ ]:
(
    features
    .withColumn("event_date", F.lit(TARGET_DATE))
    .repartition(1)
    .write
    .mode("overwrite")
    .parquet(GOLD_PATH)
)
